# **Build Your Own Image Classification in TensorFlow**

This assignment will help you guys understand the fundamentals of image classification, TensorFlow, and neural network architectures.

- GitHub: https://github.com/kaopanboonyuen/ai_for_dept_of_lands
- Author: Kao Panboonyuen


![](https://github.com/kaopanboonyuen/kaopanboonyuen.github.io/raw/main/files/MARS/MARSAIL.png)


## **Install TensorFlow with pip**

In [ ]:
# !pip install tensorflow==2.9.1 # CPU Version
# !pip install tensorflow-gpu==2.9.1 # GPU Version

## **Download Sample Dataset**

In [ ]:
!wget https://github.com/kaopanboonyuen/SC310005_ArtificialIntelligence_2023s1/raw/main/dataset/MangoLeafBD_dataset_small_v2.zip

### **Don't forget to uncomment** this line before training the model to unzip the dataset, which is currently in a zip file format.

In [ ]:
!unzip MangoLeafBD_dataset_small_v2.zip >> log.txt

# **Import Python Libraries (Deep Learning)**

In [ ]:
# ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# import system libs
import os
import time
import shutil
import pathlib
import itertools

# import data handling tools
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_style('darkgrid')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# import Deep learning Libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Activation, Dropout, BatchNormalization
from tensorflow.keras import regularizers

# **Data Preprocessing**

### **Read data and store it in dataframe**

In [ ]:
def remove_ds_store(root_dir):
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename == '.DS_Store':
                file_path = os.path.join(dirpath, filename)
                os.remove(file_path)
                print(f"Removed: {file_path}")

### **Roor Directory of Dataset**

In [ ]:
# Specify the root directory containing subfolders
root_directory = '/content/MangoLeafBD_dataset_small_v2/'

# Call the function to remove .DS_Store files
remove_ds_store(root_directory)

In [ ]:
# Generate data paths with labels
filepaths = []
labels = []

folds = os.listdir(root_directory)
for fold in folds:
    foldpath = os.path.join(root_directory, fold)
    filelist = os.listdir(foldpath)
    for file in filelist:
        fpath = os.path.join(foldpath, file)
        filepaths.append(fpath)
        labels.append(fold)

# Concatenate data paths with labels into one dataframe
Fseries = pd.Series(filepaths, name= 'filepaths')
Lseries = pd.Series(labels, name='labels')
df = pd.concat([Fseries, Lseries], axis= 1)

In [ ]:
df

### **Split dataframe into train, valid, and test**

In [ ]:
# Split df into train_df and temp_df (60% train, 40% temp)
train_df, temp_df = train_test_split(df, train_size=0.6, shuffle=True, random_state=123)

# Split temp_df into valid_df and test_df (50% valid, 50% test)
valid_df, test_df = train_test_split(temp_df, train_size=0.5, shuffle=True, random_state=123)

### **Create image data generator**

In [ ]:
# crobed image size
batch_size = 16
img_size = (224, 224)
channels = 3
img_shape = (img_size[0], img_size[1], channels)

# Recommended : use custom function for test data batch size, else we can use normal batch size.
ts_length = len(test_df)
test_batch_size = max(sorted([ts_length // n for n in range(1, ts_length + 1) if ts_length%n == 0 and ts_length/n <= 80]))
test_steps = ts_length // test_batch_size

# This function which will be used in image data generator for data augmentation, it just take the image and return it again.
def scalar(img):
    return img

tr_gen = ImageDataGenerator(preprocessing_function= scalar)
ts_gen = ImageDataGenerator(preprocessing_function= scalar)

train_gen = tr_gen.flow_from_dataframe( train_df, x_col= 'filepaths', y_col= 'labels', target_size= img_size, class_mode= 'categorical',
                                    color_mode= 'rgb', shuffle= True, batch_size= batch_size)

valid_gen = ts_gen.flow_from_dataframe( valid_df, x_col= 'filepaths', y_col= 'labels', target_size= img_size, class_mode= 'categorical',
                                    color_mode= 'rgb', shuffle= True, batch_size= batch_size)

# Note: we will use custom test_batch_size, and make shuffle= false
test_gen = ts_gen.flow_from_dataframe( test_df, x_col= 'filepaths', y_col= 'labels', target_size= img_size, class_mode= 'categorical',
                                    color_mode= 'rgb', shuffle= False, batch_size= test_batch_size)

### **Show sample from train data**

In [ ]:
g_dict = train_gen.class_indices      # defines dictionary {'class': index}
classes = list(g_dict.keys())       # defines list of dictionary's kays (classes), classes names : string
images, labels = next(train_gen)      # get a batch size samples from the generator

plt.figure(figsize= (20, 20))

for i in range(16):
    plt.subplot(4, 4, i + 1)
    image = images[i] / 255       # scales data to range (0 - 255)
    plt.imshow(image)
    index = np.argmax(labels[i])  # get image index
    class_name = classes[index]   # get class of image
    plt.title(class_name, color= 'blue', fontsize= 12)
    plt.axis('off')
plt.show()

# **Model Structure**

## Architecture 1: Simple CNN (Basic ConvNet)

This architecture is a basic CNN with convolutional layers, pooling layers, and fully connected layers.

# **Building a Simple CNN Model for Image Classification**

## **Objective:**  
In this example, we will construct a **Convolutional Neural Network (CNN)** from scratch using **TensorFlow and Keras**. The model will be used for **image classification**, incorporating convolutional and pooling layers to extract important image features.

---

### **1. Input Image Specifications**  
- **Image Dimensions:** 224x224 pixels  
- **Number of Channels:** 3 (RGB)  
- **Number of Output Classes:** Based on dataset class count  

---

### **2. Model Architecture**  

- **First Conv Layer:**  
  - **Filters:** 32  
  - **Kernel Size:** 3x3  
  - **Activation:** ReLU  
  - **Input Shape:** (224, 224, 3)  

- **First Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Second Conv Layer:**  
  - **Filters:** 64  
  - **Kernel Size:** 3x3  
  - **Activation:** ReLU  

- **Second Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Third Conv Layer:**  
  - **Filters:** 128  
  - **Kernel Size:** 3x3  
  - **Activation:** ReLU  

- **Third Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Flatten Layer:** Converts feature maps into a 1D vector  

- **Fully Connected Layer:**  
  - **Neurons:** 128  
  - **Activation:** ReLU  
  - **Dropout:** 50%  

- **Output Layer:**  
  - **Neurons:** Equal to the number of classes  
  - **Activation:** Softmax  

---

### **3. Model Compilation**  
The model is compiled using:  
- **Optimizer:** Adam (`learning_rate=0.001`)  
- **Loss Function:** Categorical Cross-Entropy  
- **Evaluation Metric:** Accuracy  

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

img_size = (224, 224)
channels = 3
img_shape = (img_size[0], img_size[1], channels)
class_count = len(list(train_gen.class_indices.keys()))

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=img_shape),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(class_count, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

## Architecture 2: Deeper CNN (More Layers)
This model is a deeper architecture with more convolutional layers to capture more complex patterns in the data.

In this example, we will construct a **Convolutional Neural Network (CNN)** from scratch using TensorFlow and Keras. The model is designed for **image classification** and follows a sequential architecture with multiple convolutional and pooling layers.

---

### **1. Input Image Specifications**  
- Input image dimensions: **224x224 pixels**  
- Number of channels: **3 (RGB)**  
- Output classes: **Based on the dataset's class count**  

---

### **2. Model Architecture**  

- **First Conv Layer:**  
  - **Filters:** 32  
  - **Kernel size:** 3x3  
  - **Activation:** ReLU  
  - **Input shape:** `(224, 224, 3)`

- **First Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Second Conv Layer:**  
  - **Filters:** 64  
  - **Kernel size:** 3x3  
  - **Activation:** ReLU  

- **Second Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Third Conv Layer:**  
  - **Filters:** 128  
  - **Kernel size:** 3x3  
  - **Activation:** ReLU  

- **Third Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Fourth Conv Layer:**  
  - **Filters:** 256  
  - **Kernel size:** 3x3  
  - **Activation:** ReLU  

- **Fourth Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Fifth Conv Layer:**  
  - **Filters:** 512  
  - **Kernel size:** 3x3  
  - **Activation:** ReLU  

- **Fifth Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Flatten Layer:** Converts the feature maps into a 1D vector  

- **Fully Connected Layer:**  
  - **Neurons:** 512  
  - **Activation:** ReLU  
  - **Dropout:** 50%  

- **Output Layer:**  
  - **Neurons:** Equal to the number of classes  
  - **Activation:** Softmax  

---

### **3. Model Compilation**  
The model is compiled using:  
- **Optimizer:** Adam (`learning_rate=0.001`)  
- **Loss Function:** Categorical Cross-Entropy  
- **Evaluation Metric:** Accuracy  

In [ ]:
# Code here

## Architecture 3: CNN with Batch Normalization (Improves Training Stability)
This architecture incorporates batch normalization layers, which help stabilize and accelerate training by normalizing the activations of the neurons.

In this example, we will build a **Convolutional Neural Network (CNN)** with **Batch Normalization** to enhance training stability and performance. The model is designed for **image classification** and follows a sequential architecture with convolutional, pooling, and fully connected layers.

---

### **1. Input Image Specifications**  
- **Image Dimensions:** 224x224 pixels  
- **Number of Channels:** 3 (RGB)  
- **Number of Output Classes:** Based on dataset class count  

---

### **2. Model Architecture**  

- **First Conv Layer:**  
  - **Filters:** 32  
  - **Kernel Size:** 3x3  
  - **Activation:** ReLU  
  - **Input Shape:** (224, 224, 3)  
  - **Batch Normalization**  

- **First Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Second Conv Layer:**  
  - **Filters:** 64  
  - **Kernel Size:** 3x3  
  - **Activation:** ReLU  
  - **Batch Normalization**  

- **Second Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Third Conv Layer:**  
  - **Filters:** 128  
  - **Kernel Size:** 3x3  
  - **Activation:** ReLU  
  - **Batch Normalization**  

- **Third Pooling Layer:**  
  - **MaxPooling:** 2x2  

- **Flatten Layer:** Converts feature maps into a 1D vector  

- **Fully Connected Layer:**  
  - **Neurons:** 256  
  - **Activation:** ReLU  
  - **Dropout:** 50%  

- **Output Layer:**  
  - **Neurons:** Equal to the number of classes  
  - **Activation:** Softmax  

---

### **3. Model Compilation**  
The model is compiled using:  
- **Optimizer:** Adam (`learning_rate=0.001`)  
- **Loss Function:** Categorical Cross-Entropy  
- **Evaluation Metric:** Accuracy

In [ ]:
# Code here

# **Assignment: Build a Simple CNN Model from Scratch**

## **Objective:**
In this assignment, you will create a simple CNN architecture from scratch. Your model will consist of:

- A convolutional layer with a 3x3 kernel
- A max-pooling layer
- Another convolutional layer with a 3x3 kernel
- Another max-pooling layer
- A fully connected (dense) layer

You will implement this using Keras and TensorFlow, **without using any pre-trained models**. By the end of this exercise, you'll understand how to design CNN architectures and use them for image classification.

---

## **Instructions:**

### **1. Input Image Dimensions:**
- Assume the input images have dimensions of `224x224` pixels and have `3` color channels (RGB).

### **2. Model Architecture:**

- **First Conv Layer:**
  - Add a convolutional layer with **32 filters** and a **kernel size of 3x3**.
  - Use the **ReLU** activation function.
  - Set the **input shape** to `(224, 224, 3)` to match the input image dimensions.

- **First Pooling Layer:**
  - Add a **max-pooling layer** with a pool size of `2x2`.

- **Second Conv Layer:**
  - Add a convolutional layer with **64 filters** and a **kernel size of 3x3**.
  - Use **ReLU** activation again.

- **Second Pooling Layer:**
  - Add another **max-pooling layer** with a pool size of `2x2`.

- **Fully Connected Layer:**
  - Add a **dense layer** with **128 neurons** and **ReLU** activation.
  - Add a **dropout layer** with a rate of **0.5** to reduce overfitting.

- **Output Layer:**
  - Add a **dense layer** with a number of units equal to the number of **classes** (for example, **10 classes**).
  - Use the **softmax** activation function for multi-class classification.

### **3. Compilation:**
- Compile the model using the **Adam optimizer**, **categorical cross-entropy loss**, and **accuracy** as the evaluation metric.

### **4. Training:**
- Train the model on a dataset of your choice. You can use any dataset you like, such as the **CIFAR-10** dataset or any custom dataset.
- Print the **accuracy** and **loss** during training.

In [ ]:
# Code here

#### **Train model**

In [ ]:
batch_size = 16   # set batch size for training
epochs = 20   # number of all epochs in training

history = model.fit(x= train_gen, epochs= epochs, verbose= 1, validation_data= valid_gen,
                    validation_steps= None, shuffle= False)

#### **Display model performance**

In [ ]:
# Define needed variables
tr_acc = history.history['accuracy']
tr_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
index_loss = np.argmin(val_loss)
val_lowest = val_loss[index_loss]
index_acc = np.argmax(val_acc)
acc_highest = val_acc[index_acc]
Epochs = [i+1 for i in range(len(tr_acc))]
loss_label = f'best epoch= {str(index_loss + 1)}'
acc_label = f'best epoch= {str(index_acc + 1)}'

# Plot training history
plt.figure(figsize= (20, 8))
plt.style.use('fivethirtyeight')

plt.subplot(1, 2, 1)
plt.plot(Epochs, tr_loss, 'r', label= 'Training loss')
plt.plot(Epochs, val_loss, 'g', label= 'Validation loss')
plt.scatter(index_loss + 1, val_lowest, s= 150, c= 'blue', label= loss_label)
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(Epochs, tr_acc, 'r', label= 'Training Accuracy')
plt.plot(Epochs, val_acc, 'g', label= 'Validation Accuracy')
plt.scatter(index_acc + 1 , acc_highest, s= 150, c= 'blue', label= acc_label)
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout
plt.show()

# **Evaluate model**

In [ ]:
ts_length = len(test_df)
test_batch_size = max(sorted([ts_length // n for n in range(1, ts_length + 1) if ts_length%n == 0 and ts_length/n <= 80]))
test_steps = ts_length // test_batch_size

train_score = model.evaluate(train_gen, steps= test_steps, verbose= 1)
valid_score = model.evaluate(valid_gen, steps= test_steps, verbose= 1)
test_score = model.evaluate(test_gen, steps= test_steps, verbose= 1)

print("Train Loss: ", train_score[0])
print("Train Accuracy: ", train_score[1])
print('-' * 20)
print("Validation Loss: ", valid_score[0])
print("Validation Accuracy: ", valid_score[1])
print('-' * 20)
print("Test Loss: ", test_score[0])
print("Test Accuracy: ", test_score[1])

# **Get Predictions**

In [ ]:
test_gen

In [ ]:
preds = model.predict(test_gen)
y_pred = np.argmax(preds, axis=1)

#### **Confusion Matrics and Classification Report**

In [ ]:
g_dict = test_gen.class_indices
classes = list(g_dict.keys())

# Confusion matrix
cm = confusion_matrix(test_gen.classes, y_pred)

plt.figure(figsize= (10, 10))
plt.imshow(cm, interpolation= 'nearest', cmap= plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()

tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes, rotation= 45)
plt.yticks(tick_marks, classes)


thresh = cm.max() / 2.
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    plt.text(j, i, cm[i, j], horizontalalignment= 'center', color= 'white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
plt.ylabel('True Label')
plt.xlabel('Predicted Label')

plt.show()

In [ ]:
# Classification report
print(classification_report(test_gen.classes, y_pred, target_names= classes))

#### **Save model**

In [ ]:
model_name = model.layers[0].name  # Get the name of the first layer
subject = 'Mango Diseases'
acc = test_score[1] * 100
save_path = ''

# Save model
save_id = str(f'{model_name}-{subject}-{"%.2f" %round(acc, 2)}.h5')
model_save_loc = os.path.join(save_path, save_id)
model.save(model_save_loc)
print(f'model was saved as {model_save_loc}')

# Save weights
weight_save_id = str(f'{model_name}-{subject}.weights.h5')
weights_save_loc = os.path.join(weight_save_id)
model.save_weights(weights_save_loc)
print(f'weights were saved as {weights_save_loc}')

#### **Generate CSV files containing classes indicies & image size**

In [ ]:
class_dict = train_gen.class_indices
img_size = train_gen.image_shape
height = []
width = []
for _ in range(len(class_dict)):
    height.append(img_size[0])
    width.append(img_size[1])

Index_series = pd.Series(list(class_dict.values()), name= 'class_index')
Class_series = pd.Series(list(class_dict.keys()), name= 'class')
Height_series = pd.Series(height, name= 'height')
Width_series = pd.Series(width, name= 'width')
class_df = pd.concat([Index_series, Class_series, Height_series, Width_series], axis= 1)
csv_name = f'{subject}-class_dict.csv'
csv_save_loc = os.path.join(save_path, csv_name)
class_df.to_csv(csv_save_loc, index= False)
print(f'class csv file was saved as {csv_save_loc}')